# 06 | Summary Table
**Final Individual Project | YRBS 2007**

## Research Question

> **Is there a significant difference in mean height among students with different levels of milk drinking frequency?**

### Analysis Variables

| Role | Variable | Definition |
|---|---|---|
| Explanatory | `NoMilkDrinking` | Milk drinking frequency, recoded into three ordered groups |
| Response | `HowTallAreYouWithoutShoesInMeters` | Self-reported height without shoes, measured in meters |
### alpha = 0.05

In [56]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from IPython.display import display, HTML

# Load dataset
df = pd.read_csv("../data/processed/YRBS_Cleaned_and_Recoded.csv")

In [58]:
# ANOVA Model
model = ols(
    "HowTallAreYouWithoutShoesInMeters ~ C(NoMilkDrinking)",
    data=df
).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

# Tukey HSD
tukey = pairwise_tukeyhsd(
    endog=df["HowTallAreYouWithoutShoesInMeters"],
    groups=df["NoMilkDrinking"],
    alpha=0.05
)

tukey_df = pd.DataFrame(
    tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)

# Labels
label_map = {
    0: "No milk drinking",
    1: "Occasional milk drinking",
    2: "Frequent milk drinking"
}

# Descriptive stats
desc = df.groupby("NoMilkDrinking")["HowTallAreYouWithoutShoesInMeters"].agg(
    ["count", "mean", "std"]
)

overall_mean = df["HowTallAreYouWithoutShoesInMeters"].mean()
overall_sd = df["HowTallAreYouWithoutShoesInMeters"].std()

# Effect size (eta squared)
ss_between = anova_table.loc["C(NoMilkDrinking)", "sum_sq"]
ss_total = anova_table["sum_sq"].sum()
eta_sq = ss_between / ss_total

if eta_sq < 0.01:
    effect_text = "Very Small Effect"
elif eta_sq < 0.06:
    effect_text = "Small Effect"
elif eta_sq < 0.14:
    effect_text = "Medium Effect"
else:
    effect_text = "Large Effect"

# ANOVA stats
k = df["NoMilkDrinking"].nunique()
N = df.shape[0]

df1 = k - 1
df2 = N - k

F = anova_table.loc["C(NoMilkDrinking)", "F"]
p = anova_table.loc["C(NoMilkDrinking)", "PR(>F)"]

p_text = "< 0.001" if p < 0.001 else f"{p:.4f}"

# Tukey formatting
tukey_lines = []
for _, row in tukey_df.iterrows():
    g1 = int(row["group1"])
    g2 = int(row["group2"])

    tukey_lines.append(
        f"{label_map[g1]} vs. {label_map[g2]} = {row['meandiff']:.4f} (Mean difference of height)"
    )

tukey_text = "\n".join(tukey_lines)

# Summary table
summary_df = pd.DataFrame({
    "Statistic": [
        "Research Question",
        "Sample Size",
        "Overall Mean Height",

        "No Milk Drinking Mean",
        "Occasional Milk Drinking Mean",
        "Frequent Milk Drinking Mean",

        "Hypotheses",
        "F-value",
        "p-value",
        "Effect Size (η²)",
        "Effect Size Magnitude",
        "Decision (α = 0.05)",
        "Conclusion",
        "Post Hoc (Tukey HSD)",
        "Interpretation"
    ],

    "Result": [
        "Is there a significant difference in mean height among milk drinking groups?",
        f"N = {N:,}",
        f"{overall_mean:.3f} m (SD = {overall_sd:.3f})",

        f"{desc.loc[0,'mean']:.3f} m (n={desc.loc[0,'count']}, SD={desc.loc[0,'std']:.3f})",
        f"{desc.loc[1,'mean']:.3f} m (n={desc.loc[1,'count']}, SD={desc.loc[1,'std']:.3f})",
        f"{desc.loc[2,'mean']:.3f} m (n={desc.loc[2,'count']}, SD={desc.loc[2,'std']:.3f})",

        "H₀: μ1 = μ2 = μ3\nH₁: At least one group mean differs",

        f"F({df1}, {df2}) = {F:.3f}",
        p_text,
        f"{eta_sq:.4f}",
        effect_text,

        "Reject H₀",

        "There is sufficient statistical evidence at the 0.05 significance level to conclude that at least one group mean height differs among the three milk drinking groups.",

        tukey_text,

        "ANOVA showed that at least one group mean height differed significantly among the groups, but it does not identify which group differs. Tukey HSD post hoc tests showed significant pairwise differences among all groups. Descriptive statistics indicated that frequent milk drinkers had the highest mean height, followed by occasional milk drinkers and non-drinkers."
 
 
    ]
})

# Title
display(HTML("""
<h3 style="text-align:left; font-weight:bold; margin-bottom:12px;">
Summary Table
</h3>
"""))

# Styling
styled_table = (
    summary_df.style
    .hide(axis="index")
    .set_table_styles([
        {
            "selector": "table",
            "props": [
                ("margin-left", "auto"),
                ("margin-right", "auto"),
                ("border-collapse", "collapse")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "left"),
                ("white-space", "pre-wrap"),
                ("padding", "6px")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("text-align", "left"),
                ("padding", "6px")
            ]
        }
    ])
)

display(HTML(styled_table.to_html()))

# Save file
summary_df.to_csv(
    "../outputs/tables/ANOVA_summary_table.csv",
    index=False,
    encoding="utf-8-sig"
)

Statistic,Result
Research Question,Is there a significant difference in mean height among milk drinking groups?
Sample Size,"N = 12,668"
Overall Mean Height,1.694 m (SD = 0.101)
No Milk Drinking Mean,"1.675 m (n=2614, SD=0.102)"
Occasional Milk Drinking Mean,"1.686 m (n=5125, SD=0.098)"
Frequent Milk Drinking Mean,"1.712 m (n=4929, SD=0.102)"
Hypotheses,H₀: μ1 = μ2 = μ3 H₁: At least one group mean differs
F-value,"F(2, 12665) = 139.056"
p-value,< 0.001
Effect Size (η²),0.0215
